In [ ]:
# Install dependencies if needed (see requirements.txt in this folder)
# !pip install --upgrade gymnasium

In [ ]:
# Install dependencies if needed (see requirements.txt in this folder)
# !pip install stable-baselines3

In [ ]:
# Make the repo-root modules (simulator.py, utils.py) importable from this folder
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

In [ ]:
import os
import torch
import pickle
import numpy as np
import multiprocessing
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

## Environment

In [ ]:
class HeatInverseEnvGPU(gym.Env):
    """
    GPU-accelerated version of HeatInverseEnv for faster RL training.
    Uses Heat2DGPU solver instead of scipy-based Heat2D.
    """

    def __init__(
        self,
        observed_dataset,
        param_ranges,
        num_sources_sensors_dist,
        Lx=2.0,
        Ly=1.0,
        nx=100,
        ny=50,
        dt=0.004,
        max_sources=2,
        max_sensors=20,
        max_steps=20,
        action_scale=0.1,
        error_threshold=1e-3,
        device=None,  # NEW: Allow device specification
        seed=None
    ):
        super().__init__()

        self.dataset = observed_dataset
        self.dt = dt
        self.param_ranges = param_ranges
        self.num_sources_sensors_dist = num_sources_sensors_dist

        # Physics domain parameters
        self.Lx = Lx
        self.Ly = Ly
        self.nx = nx
        self.ny = ny

        self.max_sources = max_sources
        self.max_sensors = max_sensors

        self.max_steps = max_steps
        self.action_scale = action_scale
        self.error_threshold = error_threshold

        # GPU device setup
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)

        print(f"HeatInverseEnvGPU initialized on device: {self.device}")

        self.rng = np.random.default_rng(seed)

        self.param_dim = self.max_sources * 3  # x,y,q for each source

        # Observation Space:
        # 1. Global Residual Stats (4): mean, std, max, norm
        # 2. Per-Sensor Mean Residuals (max_sensors): Spatial error info
        # 3. Current Parameters (param_dim)
        # 4. Source Mask (max_sources)
        # 5. Current Error (1)
        obs_dim = 4 + self.max_sensors + self.param_dim + self.max_sources + 1

        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(self.param_dim,), dtype=np.float32
        )

        # Cache solvers to avoid recreation (one per unique configuration)
        self.solver_cache = {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        idx = self.rng.integers(0, len(self.dataset))
        self.observed = self.dataset[idx]

        # Determine actual number of sensors for this sample
        self.actual_num_sensors = self.observed["sensors_xy"].shape[0]

        # Sample hypothesis config (number of sources)
        self.num_sources, _ = self._sample_num_sources()

        # Initialize mask
        self.mask = np.zeros(self.max_sources, dtype=np.float32)
        self.mask[:self.num_sources] = 1.0

        self.theta = self._random_init()
        self.prev_error, self.prev_sim = self._simulate(self.theta)

        self.step_count = 0

        return self._get_state(self.prev_sim, self.prev_error), {}

    def step(self, action):
        self.step_count += 1

        delta = action * self.action_scale

        self.theta = self._clip_params(self.theta + delta)

        error, sim = self._simulate(self.theta)

        # Reward: Improvement in error
        reward = (self.prev_error - error) * 100.0

        done = (
            self.step_count >= self.max_steps
            or error < self.error_threshold
        )

        self.prev_error = error
        self.prev_sim = sim

        state = self._get_state(sim, error)
        info = {
            "error": error,
            "theta": self.theta.copy(),
            "num_sources": self.num_sources
        }

        return state, reward, done, False, info

    # -------------------- helpers --------------------

    def _sample_num_sources(self):
        keys = list(self.num_sources_sensors_dist.keys())
        probs = list(self.num_sources_sensors_dist.values())
        n_sources = self.rng.choice([1, 2])
        return n_sources, self.actual_num_sensors

    def _get_solver(self, kappa, bc):
        """
        Get or create a cached solver for given parameters.
        This avoids recreating solvers repeatedly during training.
        """
        from simulator_gpu import Heat2DGPU

        key = (kappa, bc)
        if key not in self.solver_cache:
            self.solver_cache[key] = Heat2DGPU(
                self.Lx, self.Ly, self.nx, self.ny,
                kappa, bc=bc, device=self.device
            )
        return self.solver_cache[key]

    def _simulate(self, theta):
        """Simulate using sample metadata and Heat2DGPU solver."""
        # Extract sample metadata
        meta = self.observed["sample_metadata"]
        dt = self.dt
        nt = meta["nt"]
        kappa = meta["kappa"]
        bc = meta["bc"]
        T0 = meta["T0"]
        sensors_xy = self.observed["sensors_xy"]

        # Get cached solver
        solver = self._get_solver(kappa, bc)

        # Build sources from theta
        sources = []
        for i in range(self.num_sources):
            x, y, q = theta[3*i:3*i+3]
            sources.append({"x": float(x), "y": float(y), "q": float(q)})

        # Run simulation on GPU
        times, fields = solver.solve(dt=dt, nt=nt, T0=T0, sources=sources, store_every=1)

        # Sample at sensor positions (returns numpy array)
        sim = np.array([solver.sample_sensors(U, sensors_xy) for U in fields])

        # Compare with observed data
        observed_data = self.observed["Y_noisy"]

        # Handle potential shape mismatch
        min_len = min(sim.shape[0], observed_data.shape[0])
        sim = sim[:min_len]
        observed_data = observed_data[:min_len]

        residual = sim - observed_data
        error = np.sqrt(np.mean(residual ** 2))
        return error, sim

    def _get_state(self, sim, error):
        observed_data = self.observed["Y_noisy"]
        min_len = min(sim.shape[0], observed_data.shape[0])
        residual = sim[:min_len] - observed_data[:min_len]

        # 1. Global Stats
        stats = np.array([
            residual.mean(),
            residual.std(),
            np.max(np.abs(residual)),
            np.linalg.norm(residual)
        ])

        # 2. Per-Sensor Mean Residuals (Spatial Features)
        sensor_residuals = np.mean(residual, axis=0)

        # Pad to max_sensors
        padded_sensor_residuals = np.zeros(self.max_sensors, dtype=np.float32)
        n_sensors = min(len(sensor_residuals), self.max_sensors)
        padded_sensor_residuals[:n_sensors] = sensor_residuals[:n_sensors]

        state = np.concatenate([
            stats,
            padded_sensor_residuals,
            self.theta,
            self.mask,
            [error]
        ])
        return state.astype(np.float32)

    def _random_init(self):
        theta = np.zeros(self.param_dim, dtype=np.float32)
        for i in range(self.num_sources):
            theta[3*i+0] = self.rng.uniform(*self.param_ranges["x"])
            theta[3*i+1] = self.rng.uniform(*self.param_ranges["y"])
            theta[3*i+2] = self.rng.uniform(*self.param_ranges["q"])
        return theta

    def _clip_params(self, theta):
        clipped = theta.copy()
        for i in range(self.num_sources):
            clipped[3*i+0] = np.clip(theta[3*i+0], *self.param_ranges["x"])
            clipped[3*i+1] = np.clip(theta[3*i+1], *self.param_ranges["y"])
            clipped[3*i+2] = np.clip(theta[3*i+2], *self.param_ranges["q"])
        # Zero out unused sources
        for i in range(self.num_sources, self.max_sources):
            clipped[3*i:3*i+3] = 0.0
        return clipped

    def close(self):
        """Clean up GPU resources."""
        self.solver_cache.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# Backwards compatibility wrapper
class HeatInverseEnv(HeatInverseEnvGPU):
    """
    Alias for backwards compatibility.
    By default uses GPU if available, falls back to CPU.
    """
    pass

## Training

In [ ]:
# ---------------- load observed dataset ----------------

def load_dataset(path="data/training_dataset.pkl"):
    """Load the training dataset generated by make_training_dataset."""
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data


# ---------------- main training ----------------

def main():
    # Device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch device: {device}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Load dataset
    dataset = load_dataset()
    print(f"Loaded dataset with {len(dataset)} samples")

    param_ranges = {
        "x": (0.0, 2.0),
        "y": (0.0, 1.0),
        "q": (0.5, 2.0)
    }

    # Match the dataset distribution
    num_sources_sensors = {
        (1, 3): 0.2,
        (1, 4): 0.2,
        (2, 5): 0.6
    }

    def make_env(rank=0):
        """
        Create environment with GPU support.

        Important: Each parallel environment will share the same GPU device.
        The GPU solver caching will help minimize memory usage.
        """
        return HeatInverseEnvGPU(
            observed_dataset=dataset,
            param_ranges=param_ranges,
            num_sources_sensors_dist=num_sources_sensors,
            Lx=2.0,
            Ly=1.0,
            nx=100,
            ny=50,
            dt=0.004,
            max_sources=2,
            max_steps=10,
            action_scale=0.15,
            error_threshold=1e-3,
            device=device,  # All envs use same GPU
            seed=42 + rank  # Different seed per env
        )

    # Parallel environments configuration
    n_cores = multiprocessing.cpu_count()

    if torch.cuda.is_available():
        # With GPU: Use fewer parallel envs since GPU does heavy lifting
        # Too many parallel envs can cause GPU memory issues
        n_envs = min(8, n_cores)  # Reduced from 16 for GPU memory
        vec_env_cls = DummyVecEnv  # DummyVecEnv for GPU (avoids pickling issues)
        print(f"GPU mode: Using {n_envs} parallel environments with DummyVecEnv")
    else:
        # Without GPU: Use more parallel envs to saturate CPU
        n_envs = min(16, n_cores)
        vec_env_cls = SubprocVecEnv
        print(f"CPU mode: Using {n_envs} parallel environments with SubprocVecEnv")

    # Create vectorized environment
    env = make_vec_env(
        make_env,
        n_envs=n_envs,
        vec_env_cls=vec_env_cls
    )

    # SAC hyperparameters optimized for GPU training
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=3e-4,
        buffer_size=1_000_000,
        batch_size=512 if torch.cuda.is_available() else 256,  # Larger batches for GPU
        tau=0.005,
        gamma=0.99,
        ent_coef="auto",
        train_freq=1,
        gradient_steps=1,
        learning_starts=1_000,
        verbose=1,
        device=device,
        tensorboard_log="./tensorboard_logs/",  # Enable TensorBoard logging
    )

    # Callbacks
    checkpoint_callback = CheckpointCallback(
        save_freq=50_000,
        save_path="./checkpoints",
        name_prefix="sac_heat_gpu"
    )

    print("\n" + "="*60)
    print("Starting SAC Training with GPU Acceleration")
    print("="*60)
    print(f"Total timesteps: 200,000")
    print(f"Parallel environments: {n_envs}")
    print(f"Batch size: {512 if torch.cuda.is_available() else 256}")
    print(f"Device: {device}")
    print("="*60 + "\n")

    # Train the model
    try:
        model.learn(
            total_timesteps=200_000,
            callback=checkpoint_callback,
            log_interval=10,
            progress_bar=True  # Show progress bar
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Saving model...")
    finally:
        # Save final model
        model.save("sac_heat_inverse_gpu")
        print("Model saved to: sac_heat_inverse_gpu.zip")

        # Cleanup
        env.close()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Training finished. GPU cache cleared.")


if __name__ == "__main__":
    main()